# LeetCode #149: Max Points on a Line

https://leetcode.com/problems/max-points-on-a-line/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| Brute Force (check all triples) | O(n³) | O(1) |
| Optimal: Slope Hash Map ★ | O(n²) | O(n) |

## Understanding the Methods
### Brute Force
For every triplet of points, check if they are collinear using the cross-product formula. Track the maximum count by fixing two points and counting how many others are collinear with them.

### Optimal: Slope Hash Map ★
For each point as the anchor, compute the reduced slope (dy/dx as a fraction in lowest terms using GCD) to every other point. Use a hash map to count points sharing the same slope. Account for vertical lines and duplicate points separately. Answer is the max count + duplicates for any anchor.

## Solutions
### C#

In [ ]:
public class Solution {
    public int MaxPoints(int[][] points) {
        int n = points.Length;
        if (n <= 2) return n;
        int result = 2;
        for (int i = 0; i < n; i++) {
            var slopes = new Dictionary<(int, int), int>();
            int dups = 1, vert = 0, maxSlope = 0;
            for (int j = i + 1; j < n; j++) {
                int dx = points[j][0] - points[i][0];
                int dy = points[j][1] - points[i][1];
                if (dx == 0 && dy == 0) { dups++; continue; }
                if (dx == 0) { vert++; maxSlope = Math.Max(maxSlope, vert); continue; }
                int g = Gcd(Math.Abs(dx), Math.Abs(dy));
                if (dx < 0) { dx = -dx; dy = -dy; }
                dx /= g; dy /= g;
                var key = (dx, dy);
                slopes[key] = slopes.GetValueOrDefault(key, 0) + 1;
                maxSlope = Math.Max(maxSlope, slopes[key]);
            }
            result = Math.Max(result, maxSlope + dups);
            result = Math.Max(result, vert + dups);
        }
        return result;
    }
    private int Gcd(int a, int b) => b == 0 ? a : Gcd(b, a % b);
}

### Python

In [ ]:
from math import gcd
from collections import defaultdict
from typing import List

class Solution:
    def maxPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        if n <= 2:
            return n
        result = 2
        for i in range(n):
            slopes = defaultdict(int)
            dups = 1
            vert = 0
            max_slope = 0
            for j in range(i + 1, n):
                dx = points[j][0] - points[i][0]
                dy = points[j][1] - points[i][1]
                if dx == 0 and dy == 0:
                    dups += 1
                    continue
                if dx == 0:
                    vert += 1
                    max_slope = max(max_slope, vert)
                    continue
                g = gcd(abs(dx), abs(dy))
                if dx < 0:
                    dx, dy = -dx, -dy
                key = (dx // g, dy // g)
                slopes[key] += 1
                max_slope = max(max_slope, slopes[key])
            result = max(result, max_slope + dups, vert + dups)
        return result

### Go

In [ ]:
package main

import "fmt"

func gcd(a, b int) int {
    if b == 0 {
        return a
    }
    return gcd(b, a%b)
}

func maxPoints(points [][]int) int {
    n := len(points)
    if n <= 2 {
        return n
    }
    result := 2
    for i := 0; i < n; i++ {
        type key struct{ dx, dy int }
        slopes := map[key]int{}
        dups, vert, maxSlope := 1, 0, 0
        for j := i + 1; j < n; j++ {
            dx := points[j][0] - points[i][0]
            dy := points[j][1] - points[i][1]
            if dx == 0 && dy == 0 {
                dups++
                continue
            }
            if dx == 0 {
                vert++
                if vert > maxSlope {
                    maxSlope = vert
                }
                continue
            }
            g := gcd(abs(dx), abs(dy))
            if dx < 0 {
                dx, dy = -dx, -dy
            }
            k := key{dx / g, dy / g}
            slopes[k]++
            if slopes[k] > maxSlope {
                maxSlope = slopes[k]
            }
        }
        if maxSlope+dups > result {
            result = maxSlope + dups
        }
        if vert+dups > result {
            result = vert + dups
        }
    }
    return result
}

func abs(x int) int {
    if x < 0 {
        return -x
    }
    return x
}

func main() {
    fmt.Println(maxPoints([][]int{{1,1},{2,2},{3,3}})) // 3
}

### Rust

In [ ]:
use std::collections::HashMap;

pub struct Solution;

impl Solution {
    pub fn max_points(points: Vec<Vec<i32>>) -> i32 {
        let n = points.len();
        if n <= 2 { return n as i32; }
        let mut result = 2i32;
        for i in 0..n {
            let mut slopes: HashMap<(i64, i64), i32> = HashMap::new();
            let (mut dups, mut vert, mut max_slope) = (1i32, 0i32, 0i32);
            for j in (i+1)..n {
                let dx = (points[j][0] - points[i][0]) as i64;
                let dy = (points[j][1] - points[i][1]) as i64;
                if dx == 0 && dy == 0 { dups += 1; continue; }
                if dx == 0 { vert += 1; max_slope = max_slope.max(vert); continue; }
                let g = Self::gcd(dx.abs(), dy.abs());
                let (ndx, ndy) = if dx < 0 { (-dx/g, -dy/g) } else { (dx/g, dy/g) };
                let cnt = slopes.entry((ndx, ndy)).or_insert(0);
                *cnt += 1;
                max_slope = max_slope.max(*cnt);
            }
            result = result.max(max_slope + dups).max(vert + dups);
        }
        result
    }
    fn gcd(a: i64, b: i64) -> i64 { if b == 0 { a } else { Self::gcd(b, a % b) } }
}

## Examples

**Common:** `points = [[1,1],[2,2],[3,3]]` → `3` (all three on y=x)

**Slightly Complex:** `points = [[1,1],[3,2],[5,3],[4,1],[2,3],[1,4]]` → `4` (slope 1/2 line)

**Edge Time:** `points = [[0,0]]` → `1` (single point)

**Edge Space:** `points = [[0,0],[0,0],[0,0]]` → `3` (all duplicates)

**Almost-Impossible:** `points = [[0,0],[1,0],[0,1],[1,1]]` → `2` (no 3 collinear in a square)